In [1]:
# ============================================================
# FINAL DATASET
# 100 Topics × 3 Production Conditions = 300 Texts
# ============================================================

import pandas as pd
import os

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------

HUMAN_FILE = "/kaggle/input/datasets/tanjimshaharia/human-articles-100/human_articles_100.csv"

LLM_FILE = "/kaggle/input/datasets/tanjimshaharia/llmbnllm/llm_bangla_articles.csv"

MT_FILE = "/kaggle/input/datasets/tanjimshaharia/mttbnmtt/mtt_bangla_articles.csv"

OUTPUT_FILE = "/kaggle/working/final_bangla_stylometry_300.csv"

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

human = pd.read_csv(HUMAN_FILE)
llm = pd.read_csv(LLM_FILE)
mt = pd.read_csv(MT_FILE)

print("Human:", human.shape)
print("LLM:", llm.shape)
print("MT:", mt.shape)

# ------------------------------------------------------------
# KEEP ONLY WHAT WE NEED
# ------------------------------------------------------------

human = human[
    [
        "topic_id",
        "topic",
        "selected_passage"
    ]
].rename(
    columns={
        "selected_passage": "human_bangla"
    }
)

llm = llm[
    [
        "topic_id",
        "topic",
        "llm_bangla"
    ]
]

mt = mt[
    [
        "topic_id",
        "topic",
        "english_passage",
        "mt_bangla"
    ]
]

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

final = human.merge(
    llm,
    on=["topic_id", "topic"],
    how="inner"
)

final = final.merge(
    mt,
    on=["topic_id", "topic"],
    how="inner"
)

# ------------------------------------------------------------
# SORT
# ------------------------------------------------------------

final = final.sort_values(
    "topic_id"
).reset_index(drop=True)

# ------------------------------------------------------------
# SAVE WIDE FORMAT
# ------------------------------------------------------------

final.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# ALSO CREATE LONG FORMAT
# ------------------------------------------------------------

long = pd.DataFrame({
    "topic_id": list(final["topic_id"]) * 3,
    "topic": list(final["topic"]) * 3,
    "condition": (
        ["human"] * len(final)
        + ["llm"] * len(final)
        + ["machine_translation"] * len(final)
    ),
    "text": (
        list(final["human_bangla"])
        + list(final["llm_bangla"])
        + list(final["mt_bangla"])
    )
})

LONG_OUTPUT = "/kaggle/working/final_bangla_stylometry_long_300.csv"

long.to_csv(
    LONG_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL DATASET CREATED")
print("=" * 60)

print("Topics:", len(final))
print("Texts:", len(long))

print("\nCondition counts:")
print(long["condition"].value_counts())

print("\nWide dataset:")
print(final.shape)

print("\nLong dataset:")
print(long.shape)

print("\nFiles:")
print(OUTPUT_FILE)
print(LONG_OUTPUT)

print("\nFirst few rows:")
print(final.head(3))

Human: (100, 11)
LLM: (100, 9)
MT: (100, 17)

FINAL DATASET CREATED
Topics: 100
Texts: 300

Condition counts:
condition
human                  100
llm                    100
machine_translation    100
Name: count, dtype: int64

Wide dataset:
(100, 6)

Long dataset:
(300, 4)

Files:
/kaggle/working/final_bangla_stylometry_300.csv
/kaggle/working/final_bangla_stylometry_long_300.csv

First few rows:
   topic_id                          topic  \
0         1    Traffic congestion in Dhaka   
1         2    Air pollution in Bangladesh   
2         3  Water pollution in Bangladesh   

                                        human_bangla  \
0  এ কারণে যানজটের সৃষ্টি হয়। এ ছাড়া ফৌজদারহাট বা...   
1  বাতাসের এই মান ‘গ্রহণযোগ্য’ বা ‘মাঝারি’ মানের ...   
2  গবেষণায় দেখা গেছে, কেরানীগঞ্জের হযরতপুর ইউনিয়ন...   

                                          llm_bangla  \
0  রাজধানী ঢাকায় তীব্র যানজট দিন দিন একটি অসহনীয় ...   
1  বায়ুুদূষণের চরম সংকটের মুখোমুখি বাংলাদেশ। বিশ...   
2  বাংলাদেশে পানিদূষণ